[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manzt/anywidget/blob/main/docs/public/notebooks/counter.ipynb)

In [1]:
%%capture
%pip install --quiet anywidget

This example demonstrates how to synchronize model state between the widget frontend and Python kernel with **anywidget**. 

The `render` function creates a `<button>` element and registers an event handler to increment the model `count` when the button is clicked. A second event handler is registered to update the text output each time `count` changes on the model.

In [2]:
import anywidget
import traitlets


class CounterWidget(anywidget.AnyWidget):
    _esm = """
    function render({ model, el }) {
      let count = () => model.get("count");
      let btn = document.createElement("button");
      btn.classList.add("counter-button");
      btn.innerHTML = `count is ${count()}`;
      btn.addEventListener("click", () => {
        model.set("count", count() + 1);
        model.save_changes();
      });
      model.on("change:count", () => {
        btn.innerHTML = `count is ${count()}`;
      });
      el.appendChild(btn);
    }
    export default { render };
    """
    _css = """
    .counter-button {
      background-image: linear-gradient(to right, #a1c4fd, #c2e9fb);
      border: 0;
      border-radius: 10px;
      padding: 10px 50px;
      color: white;
    }
    """
    count = traitlets.Int(0).tag(sync=True)


w = CounterWidget()
w.count = 60

w

By treating the model as the source of truth, whether Python kernel or JavaScript update count, the count displayed is correct. Additionally, a single model serves as the source of truth for all _views_ of that model. Therefore when `w` is displayed in another cell, the view is synchronized with the the output cell above.

In [3]:
w

But the state of a new widget instance is independent,

In [4]:
CounterWidget()